# Phase 3 — fine-tune

**33.2 sec/kimg** on 2x T4 with `batch_gpu=32`, so 300 kimg is **2.8 h** per class.
Each class has its own **train** and **results** cell, so you can run one, look,
and decide before spending the next.

Every class trains from the **LSUN Dog net**, never from another class — separate
dataset, separate output, separate model. No mixing.

## Run as a saved version

> **Save Version → Save & Run All (Commit)**

A browser tab will not survive this. **GPU T4 x2**, **Internet On**,
`stylegan.zip` attached. All three classes is 8.3 h, inside the 12 h cap.

To run just one, run steps 1–2 then that class's two cells.

## 1. Settings

In [ ]:
KIMG      = 300   # 2.8 h per class
GPUS      = 2     # 2.14x faster than 1; needs the last two patches in step 2
BATCH_GPU = 32    # 64 // (32 * 2) = 1 accumulation round
FREEZED   = 0     # FreezeD. First knob to try if ada_p climbs past ~0.7.

print(f"{KIMG} kimg = {KIMG * 33.2 / 3600:.1f} h per class")

## 2. Setup

Seven patches for torch 2.x; `patch()` asserts each target exists, so a silent
no-op is impossible. Reasoning in `docs/stylegan.md`.

The last two matter only for `GPUS=2`.

In [ ]:
import os, sys, json, time, pathlib, subprocess, shutil
import torch

REPO = "/kaggle/working/stylegan2-ada-pytorch"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "-q",
                "https://github.com/NVlabs/stylegan2-ada-pytorch.git", REPO], check=True)
sys.path.insert(0, REPO)
NL = chr(10)

def patch(rel, old, new):
    f = pathlib.Path(REPO) / rel
    s = f.read_text()
    assert old in s, f"patch target not found in {rel}"
    f.write_text(s.replace(old, new))

# Kernels report "Failed!" after building fine.
patch("torch_utils/custom_ops.py",
      "torch.utils.cpp_extension.load(name=module_name",
      "module = torch.utils.cpp_extension.load(name=module_name")
patch("torch_utils/custom_ops.py",
      "        module = importlib.import_module(module_name)" + NL, "")

# TypeError: object.__init__() takes exactly one argument
patch("torch_utils/misc.py", "super().__init__(dataset)", "super().__init__()")

# R1 needs grid_sample's second derivative, which torch still lacks.
GSG = "torch_utils/ops/grid_sample_gradfix.py"
patch(GSG, "any(torch.__version__.startswith(x) for x in ['1.7.', '1.8.', '1.9'])",
      "True")
patch(GSG,
      "op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')" + NL +
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False)",
      "op = torch.ops.aten.grid_sampler_2d_backward" + NL +
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False, [True, True])")

# batch_gpu is pinned to mb // 8 (NVlabs' rig, not ours). Default unchanged.
patch("train.py", "args.batch_gpu = spec.mb // spec.ref_gpus",
      "args.batch_gpu = int(os.environ.get('BATCH_GPU', spec.mb // spec.ref_gpus))")

# Multi-GPU only: ranks disagree on noise_const. Sync once from rank 0.
patch("training/training_loop.py",
      "    # Print network summary tables.",
      NL.join(["    if num_gpus > 1:",
               "        torch.cuda.set_device(device)",
               "        for _m in [G, D, G_ema]:",
               "            for _, _t in misc.named_params_and_buffers(_m):",
               "                torch.distributed.broadcast(_t, src=0)",
               "",
               "    # Print network summary tables."]))

DATA = next(p.parent for p in pathlib.Path("/kaggle/input").glob("**/summary.json"))
URL = ("https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/"
       "transfer-learning-source-nets/lsundog-res256-paper256-kimg100000-noaug.pkl")
PKL = "/kaggle/working/lsundog-res256.pkl"
if not os.path.exists(PKL):
    subprocess.run(["wget", "-q", "-O", PKL, URL], check=True)

NOISE = ("conv2d_gradfix", "Grad strides", "grad.sizes()", "bucket_view.sizes()")

def train(cls, kimg=None, resume=None):
    """Fine-tune one class. resume=None starts from LSUN Dog, so classes never
    inherit each other's weights."""
    kimg = kimg or KIMG
    zp = f"/kaggle/working/{cls}.zip"
    if not os.path.exists(zp):
        subprocess.run([sys.executable, f"{REPO}/dataset_tool.py",
                        f"--source={DATA / cls}", f"--dest={zp}"], check=True)

    cmd = [sys.executable, f"{REPO}/train.py",
           f"--outdir=/kaggle/working/{cls}_run", f"--data={zp}", f"--gpus={GPUS}",
           "--cfg=paper256", "--mirror=1", "--aug=ada", "--target=0.6",
           f"--resume={resume or PKL}", "--snap=10", "--metrics=none", f"--kimg={kimg}"]
    if FREEZED:
        cmd.append(f"--freezed={FREEZED}")

    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1,
                         env=dict(os.environ, BATCH_GPU=str(BATCH_GPU)))
    for line in p.stdout:
        if not any(n in line for n in NOISE):
            print(line, end="")
    p.wait()
    print(f"{cls}: exit {p.returncode}, {(time.time()-t0)/3600:.2f} h")

def show(cls):
    """Grids at start / middle / end, plus the loss and ada_p trace."""
    import matplotlib.pyplot as plt
    import PIL.Image

    dirs = sorted(pathlib.Path(f"/kaggle/working/{cls}_run").glob("00000-*"))
    if not dirs:
        print(f"no run found for {cls}")
        return
    run = dirs[-1]
    # fakes_init.png must be excluded: "_" sorts AFTER digits, so a plain
    # glob puts the dog initialisation last and hides the final grid.
    grids = sorted(run.glob("fakes[0-9]*.png"))
    snaps = sorted(run.glob("network-snapshot-*.pkl"))

    for g in [grids[0], grids[len(grids) // 2], grids[-1]]:
        im = PIL.Image.open(g)
        im.thumbnail((1100, 1100))
        plt.figure(figsize=(13, 13 * im.height / im.width))
        plt.imshow(im); plt.axis("off"); plt.title(f"{cls} - {g.name}"); plt.show()

    print(f"{cls}: {len(snaps)} snapshots, "
          f"{sum(s.stat().st_size for s in snaps)/1e9:.1f} GB")
    ticks = [json.loads(l) for l in (run / "stats.jsonl").read_text().splitlines() if l.strip()]
    get = lambda t, k: t.get(k, {}).get("mean", 0)
    for t in ticks[::max(1, len(ticks) // 8)]:
        print(f"  kimg {get(t,'Progress/kimg'):6.0f}  G {get(t,'Loss/G/loss'):7.3f}"
              f"  D {get(t,'Loss/D/loss'):7.3f}  ada_p {get(t,'Progress/augment'):.3f}")

print("torch", torch.__version__, "|", torch.cuda.device_count(), "gpus | 7 patches applied")

## 3. Mammalian — 789 images, 37% of the target

Trains from LSUN Dog, ~2.8 h. To extend it later:
`train("mammalian", kimg=200, resume="<snapshot path>")`

In [ ]:
train("mammalian")

In [ ]:
show("mammalian")

## 4. Arthropod — 237 images, 11% of the target

Trains from LSUN Dog, ~2.8 h. To extend it later:
`train("arthropod", kimg=200, resume="<snapshot path>")`

In [ ]:
train("arthropod")

In [ ]:
show("arthropod")

## 5. Plant Fungus — 197 images, 9% of the target

Trains from LSUN Dog, ~2.8 h. To extend it later:
`train("plant_fungus", kimg=200, resume="<snapshot path>")`

In [ ]:
train("plant_fungus")

In [ ]:
show("plant_fungus")

## Done — then what

1. **Look at the last grid.** Do they read as creatures of this class, and are
   they *varied*? Sharp but samey means memorisation, not success.
2. If `ada_p` passed ~0.7, the discriminator was straining — raise `--target`
   or try FreezeD.
3. The remaining seven classes cost ~19 h, so all ten fit one ~30 GPU-h week.

Snapshots are in this notebook's **Output**, ~350 MB each. Keep the best per class.